# P104 — WebArena: un entorno web realista para construir agentes autónomos

## 1. Título y paper

**Paper:** *WebArena: A Realistic Web Environment for Building Autonomous Agents*  
**Autoría:** Shuyan Zhou, Frank F. Xu, Hao Zhu, Xuhui Zhou, Robert Lo, y otros  
**Año y venue:** 2023 · arXiv:2307.13854 · ICLR 2024  
**Nivel:** L3 · **Motor:** `webarena`  
**Ficha completa:** [`P104_webarena`](../../papers/foundational/P104_webarena/README.md)

**Hito:** Evalúa agentes de navegador comprobando el ESTADO del sitio al terminar, no lo que el agente dice haber hecho.

- [arXiv:2307.13854](https://arxiv.org/abs/2307.13854)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los agentes de navegador se evaluaban con capturas, con juicios de un modelo o con el propio informe del agente. Un agente elocuente puntuaba alto sin haber completado la tarea, y los resultados no eran comparables entre trabajos.
2. Ejecutar una implementación mínima de la propuesta: Un entorno reproducible con sitios reales autoalojados —comercio, foro, repositorio, gestor de contenidos— y, para cada tarea, un verificador programático que inspecciona el estado final del sitio.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P51
- P62
- Shi et al. (2017), World of Bits


## 4. Intuición

Un agente de navegador dice que ha completado la tarea. ¿Lo ha hecho? Preguntárselo no es una evaluación, y mirar una captura tampoco. WebArena responde comprobando el **estado del sitio**: si el pedido se hizo, existe el pedido.


## 5. Concepto mínimo

```text
Evaluación por autoinforme    →  mide la elocuencia del agente
Evaluación por captura        →  mide si la pantalla parece correcta
Evaluación FUNCIONAL          →  un script inspecciona el estado final del sitio

Sitios reales autoalojados · estado reiniciable · un verificador por tarea
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('webarena', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿En cuántas tareas dice el agente haber terminado?
2. ¿En cuántas lo confirma la verificación del estado final?
3. ¿Qué tipo de tarea falla?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('webarena', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('webarena', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El agente declara haber terminado en **8 de 8** tareas, con una confianza media de 0,793. La verificación dice que acertó en **4**. Por tipo: informacion 0,667 · navegacion 1,0 · **transaccion 0,0**. Y el exceso medio de pasos es 5,12.


## 10. Comentario pedagógico

Las tareas que fallan son justamente las que **cambian el estado** del sitio. Consultar es fácil; comprar, publicar o modificar exige mantener el objetivo a través de varios pasos irreversibles. Y es exactamente la clase de tarea que interesa automatizar, así que la brecha no es un detalle del banco de pruebas.


## 11. Error o anti-patrón deliberado

Anti-patrón: dejar que un agente decida si ha terminado.


In [ ]:
print('Un agente que dice «listo» con confianza 0,95 y no hizo nada puntua igual')
print('que uno que lo hizo, si la evaluacion es su propio informe.')
print('Verificar el estado final cuesta escribir un comprobador por tarea. Ese es el precio.')

## 12. Corrección

La diferencia entre declarado y verificado:


In [ ]:
r = run_paper_lab('webarena', seed=7)['result']
print('declara haber terminado:', r['el_agente_declara_haber_terminado'])
print('verificado por estado  :', r['exito_verificado_por_estado_final'])
print('por tipo               :', r['por_tipo_de_tarea'])
print('exceso medio de pasos  :', r['exceso_medio_de_pasos'])

## 13. Desafío guiado

Explica por qué el exceso de pasos importa aunque la tarea acabe bien, y qué protege un límite de pasos.


In [ ]:
r = run_paper_lab('webarena', seed=3)['result']
show(r)

## 14. Desafío autónomo

Define tres tareas de un sitio que uses, escribe su verificador programático y ejecútalas con un agente. Documenta la diferencia entre lo que el agente dice y lo que el verificador confirma.


## 15. Evidencia de aprendizaje

Guarda la comparación entre autoinforme y verificación, con la tasa por tipo de tarea.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P104_webarena/README.md) · evaluación formal: [`assessments/papers/P104_webarena.md`](../../assessments/papers/P104_webarena.md)


## 16. Cierre

Para operar un navegador hay que poder señalar los elementos. Y muchos elementos no tienen texto que leer.


## 17. Conexión con el siguiente hito

- P105
- P106

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
